In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('/content/nasa_power_global_32countries.csv')

# Melt the DataFrame to long format
df_melted = df.melt(
    id_vars=['date', 'country', 'lat', 'lon'],
    var_name='observation_date',
    value_name='value'
)

# Convert 'observation_date' to datetime objects, coercing errors
df_melted['observation_date'] = pd.to_datetime(df_melted['observation_date'], format='%Y%m%d', errors='coerce')

# Convert 'value' to numeric, coercing errors
df_melted['value'] = pd.to_numeric(df_melted['value'], errors='coerce')

# Drop rows where 'observation_date' or 'value' could not be converted
df_melted.dropna(subset=['observation_date', 'value'], inplace=True)

# Sort the data by location and date
df_melted.sort_values(by=['country', 'lat', 'lon', 'observation_date'], inplace=True)

# Display the first few rows and check for missing values after melting and conversion
display(df_melted.head())
display(df_melted.isnull().sum())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

# Prepare data for Random Forest
# Create time-based features
df_melted['year'] = df_melted['observation_date'].dt.year
df_melted['month'] = df_melted['observation_date'].dt.month
df_melted['day'] = df_melted['observation_date'].dt.day
df_melted['dayofweek'] = df_melted['observation_date'].dt.dayofweek
df_melted['dayofyear'] = df_melted['observation_date'].dt.dayofyear


# One-hot encode the 'date' (climate variable) column for Random Forest
df_rf = pd.get_dummies(df_melted, columns=['date'], drop_first=True)

# Define features (X) and target (y) for Random Forest
# Exclude original date and observation_date columns, and the target variable itself
X_rf = df_rf.drop(['observation_date', 'value', 'country', 'lat', 'lon'], axis=1)
y_rf = df_rf['value']

# Chronological split for Random Forest (e.g., 80% train, 20% test)
split_date = df_rf['observation_date'].quantile(0.8)
X_train_rf = X_rf[df_rf['observation_date'] <= split_date]
X_test_rf = X_rf[df_rf['observation_date'] > split_date]
y_train_rf = y_rf[df_rf['observation_date'] <= split_date]
y_test_rf = y_rf[df_rf['observation_date'] > split_date]


# Prepare data for Prophet
# Prophet requires columns named 'ds' (datetime) and 'y' (target)
df_prophet = df_melted[['observation_date', 'value']].rename(columns={'observation_date': 'ds', 'value': 'y'})

# Chronological split for Prophet (using the same split date as RF for consistency)
train_prophet = df_prophet[df_melted['observation_date'] <= split_date]
test_prophet = df_prophet[df_melted['observation_date'] > split_date]

# Prepare data for LSTM
# LSTM typically works with sequences of data. This requires more complex restructuring,
# but for this subtask, we will focus on the required split and scaling.
# We will scale the 'value' column
scaler = MinMaxScaler()
df_lstm = df_melted.copy()
df_lstm['value_scaled'] = scaler.fit_transform(df_lstm[['value']])

# Chronological split for LSTM (using the same split date)
train_lstm = df_lstm[df_lstm['observation_date'] <= split_date]
test_lstm = df_lstm[df_lstm['observation_date'] > split_date]

# Display the shapes of the training and testing sets for each model
print("Random Forest Data Shapes:")
print("X_train_rf:", X_train_rf.shape)
print("X_test_rf:", X_test_rf.shape)
print("y_train_rf:", y_train_rf.shape)
print("y_test_rf:", y_test_rf.shape)

print("\nProphet Data Shapes:")
print("train_prophet:", train_prophet.shape)
print("test_prophet:", test_prophet.shape)

print("\nLSTM Data Shapes:")
print("train_lstm:", train_lstm.shape)
print("test_lstm:", test_lstm.shape)